# ISIC 2024 `strict_input + iddx_full` Dataset Contract

이 노트북은 ISIC 2024 `train-metadata.csv`를 바탕으로, tabular/fusion model 학습에서 사용할 `strict_input`과 같은 sample의 `iddx_full`을 안전하게 추적하기 위한 데이터셋 계약 문서다.

현재 논문 방향의 기본 입력은 `image + strict_input`이다. `iddx_full` 및 그 파생값은 ordinary inference-time feature가 아니며, 후보 실험에서만 사용할 수 있는 train-only privileged supervision 신호다.

## 0. 개요

### 0.1 이 노트북의 범위

1. 데이터와 누수 위험 요약
2. 컬럼 role registry
3. `strict_input` 계약
4. `iddx_full` train-only 계약
5. Label Embedding / Graph Metric Learning 후보 전처리 계약
6. 논문용 split terminology와 Triple Stratified CV 계약
7. split / preprocessing leakage contract
8. handoff checklist

### 0.2 최종 규칙

- `strict_input`은 `cv_validation`/`test_data`/inference에서 요구할 수 있는 ordinary metadata model input이다.
- `iddx_full`은 같은 `isic_id` row에서 추적하되, train-only candidate branch에서만 사용한다.
- `iddx_label_group` 같은 `iddx_full` 파생값은 `cv_validation`/`test_data` scoring 또는 inference에 들어가지 않는다.
- `isic2024_official_train_metadata`에서 patient-level `train_validation_data`와 `test_data`를 먼저 분리한다.
- `train_validation_data` 내부에서만 `cv_train` / `cv_validation` 5-fold CV를 구성한다.
- 최종 test라는 용어는 처음부터 잠근 `test_data`에만 사용한다.
- split은 `patient_id -> lesion_id -> isic_id` 구조를 보존해야 한다.
- 데이터 분포에서 학습되는 preprocessing state는 `cv_train` 또는 `final_train`에서만 fit한다.

### 0.3 이 노트북에서 하지 않는 것

이 파일은 EDA 전체나 모델링 노트북이 아니다. 다음 작업은 이 노트북의 active scope에서 제외한다.

- feature engineering 설계
- PCA/VIF 기반 변수 선택
- winsorization, log1p, robust scaling 같은 preprocessing artifact 저장
- tail enrichment 기반 변수 판단
- 모델 학습 또는 threshold 선택
- split CSV / processed CSV 파일 저장

필요하면 위 항목은 별도 CLI 또는 실험 노트북에서 다룬다. 이 파일은 `strict_input + iddx_full` dataset contract, split terminology, leakage boundary를 고정하는 데 집중한다.

### 0.4 용어 통일

- `isic2024_official_train_metadata`: `/data/raw/train-metadata.csv`. 실제 논문용 split의 source이며, 여기에서 `train_validation_data`와 `test_data`를 만든다.
- `isic2024_official_test_metadata`: `/data/raw/test-metadata.csv`. public test metadata이며 sample 수가 3개뿐이므로 schema 확인 전용으로만 사용한다.
- `train_validation_data`: `test_data`를 제외한 나머지 patient-level data. 모델 학습, hyperparameter 선택, ablation, CV validation에 사용한다.
- `test_data`: `isic2024_official_train_metadata`에서 처음 분리해 잠가 두는 fixed holdout test. 모델 선택, feature 선택, threshold 선택, early stopping에 사용하지 않는다.
- `cv_train`: `train_validation_data` 내부 5-fold CV에서 각 fold의 학습 split.
- `cv_validation`: `train_validation_data` 내부 5-fold CV에서 각 fold의 검증 split.
- `final_train`: 최종 설정 확정 후 `test_data` 평가 직전에 학습하는 데이터. 기본값은 전체 `train_validation_data`다.
- `strict_input`: 추론 시점에 사용 가능한 ordinary tabular metadata model input.
- `strict_input_numeric`: `strict_input` 안의 원시 수치형 metadata 컬럼.
- `strict_input_categorical`: `strict_input` 안의 원시 범주형 metadata 컬럼.
- `iddx_full_train_text`: 같은 `isic_id` sample에 연결되는 `iddx_full` 원문 또는 정규화 텍스트.
- `iddx_label_group`: train-only candidate에서 `iddx_full`을 수동 그룹화한 `MEL/BCC/SCC/NV/OTHER` label.
- `reference_only`: 일반 inference input으로 쓰지 않는 leakage-risk reference field 묶음.
- `inference contract`: 최종 예측 시 `iddx_full` 없이 `image + strict_input` 또는 `strict_input`만 요구해야 한다는 입력 계약.

## 1. 데이터와 위험 요약

이 장은 계약 문서에 필요한 최소 데이터 사실만 확인한다. 원자료는 읽기 전용으로 다루고, 이 노트북은 산출물을 저장하지 않는다.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'notebooks/isic_2024').exists()
        and (candidate / 'src/isic2024_multimodal').exists()
    ),
    Path.cwd(),
)

METADATA_DIR_CANDIDATES = [
    PROJECT_ROOT / 'data/raw/isic_2024_challenge',
    PROJECT_ROOT / 'data/raw',
]
METADATA_DIR = next(
    (
        candidate
        for candidate in METADATA_DIR_CANDIDATES
        if (candidate / 'train-metadata.csv').exists()
        and (candidate / 'test-metadata.csv').exists()
    ),
    None,
)
if METADATA_DIR is None:
    checked_paths = '\n'.join(str(path) for path in METADATA_DIR_CANDIDATES)
    raise FileNotFoundError(
        'Could not find ISIC 2024 metadata files. Expected train-metadata.csv '
        f'and test-metadata.csv under one of:\n{checked_paths}'
    )

TRAIN_PATH = METADATA_DIR / 'train-metadata.csv'
PUBLIC_TEST_PATH = METADATA_DIR / 'test-metadata.csv'
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
public_test_df = pd.read_csv(PUBLIC_TEST_PATH, low_memory=False)
df = train_df.copy()
target_name_map = {0: 'benign', 1: 'malignant'}

summary_df = pd.DataFrame(
    [
        {'item': 'train_rows', 'value': len(train_df)},
        {'item': 'train_patients', 'value': train_df['patient_id'].nunique()},
        {'item': 'train_lesion_id_non_null', 'value': int(train_df['lesion_id'].notna().sum())},
        {'item': 'positive_rows', 'value': int(train_df['target'].sum())},
        {'item': 'positive_rate_pct', 'value': round(float(train_df['target'].mean() * 100), 5)},
        {'item': 'public_test_rows', 'value': len(public_test_df)},
    ]
)

patient_target_summary_df = train_df.groupby('patient_id').agg(
    n_rows=('isic_id', 'size'),
    positive_rows=('target', 'sum'),
    lesion_id_non_null=('lesion_id', lambda s: int(s.notna().sum())),
).reset_index()
patient_target_summary_df['has_malignant'] = patient_target_summary_df['positive_rows'] > 0
patient_risk_summary_df = pd.DataFrame(
    [
        {'item': 'patients_with_malignant', 'value': int(patient_target_summary_df['has_malignant'].sum())},
        {'item': 'max_rows_per_patient', 'value': int(patient_target_summary_df['n_rows'].max())},
        {'item': 'median_rows_per_patient', 'value': round(float(patient_target_summary_df['n_rows'].median()), 3)},
    ]
)

display(Markdown('**표 1-a. 데이터 규모와 target 희귀성**'))
display(summary_df)
display(Markdown('**표 1-b. patient-level grouping 위험 요약**'))
display(patient_risk_summary_df)

**표 1-a. 데이터 규모와 target 희귀성**

,item,value
0,train_rows,401059.00000
1,train_patients,1042.00000
2,train_lesion_id_non_null,22058.00000
3,positive_rows,393.00000
4,positive_rate_pct,0.09799
5,public_test_rows,3.00000


**표 1-b. patient-level grouping 위험 요약**

,item,value
0,patients_with_malignant,259.0
1,max_rows_per_patient,9184.0
2,median_rows_per_patient,241.5


### 1.1 해석

train metadata는 401,059 rows와 1,042 patients로 구성되며, malignant positive는 393 rows뿐이다. positive rate는 0.09799%로 ultra-rare target에 해당한다.

row 수에 비해 patient 수가 작고, patient별 row 수의 median은 241.5, 최대값은 9,184이다. 따라서 row는 독립 표본이 아니라 `patient_id` 단위 cluster 구조를 가진다. paper-facing 실험에서 row-level random split은 허용하지 않고, 반드시 patient-level split을 사용해야 한다.

public `test-metadata.csv`는 schema 확인 정도에만 사용하고, EDA/feature choice/model selection의 근거로 사용하지 않는다.

## 2. 컬럼 role registry

이 장은 각 컬럼이 어떤 regime에 속하는지 고정한다. 목적은 입력 후보를 늘리는 것이 아니라, inference-time input과 train-only/reference-only 신호를 분리하는 것이다.

In [9]:
REGIME_DISPLAY_MAP = {
    'strict': 'strict_input',
    'schema_constant': 'Schema-only constant metadata / excluded from model input',
    'relaxed': 'Relaxed context optional sensitivity check',
    'oracle': 'Train-only iddx_full text',
    'reference': 'Reference only / excluded reference fields',
    'not_feature': 'Identifier / grouping key',
    'label': 'Label',
}

RELAXED_EXTRA_COLS = ['attribution', 'copyright_license']
TRAIN_ONLY_IDDX_FULL_TEXT_COLS = ['iddx_full']
REFERENCE_ONLY_COLS = [
    'lesion_id', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5',
    'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence',
]
SCHEMA_ONLY_CONSTANT_COLS = ['image_type']
ORDINARY_METADATA_COLS = [
    'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm',
    'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B',
    'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext',
    'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL',
    'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location',
    'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM',
    'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt',
    'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',
]
STRICT_COLS = [column for column in ORDINARY_METADATA_COLS if column not in SCHEMA_ONLY_CONSTANT_COLS]

CATEGORY_MAP = {
    'label': ['target'],
    'identifier_grouping': ['isic_id', 'patient_id'],
    'strict_input': STRICT_COLS,
    'schema_only_constant_metadata': SCHEMA_ONLY_CONSTANT_COLS,
    'relaxed_context_optional': RELAXED_EXTRA_COLS,
    'train_only_iddx_full_text': TRAIN_ONLY_IDDX_FULL_TEXT_COLS,
    'reference_only_excluded': REFERENCE_ONLY_COLS,
}


def assign_regime(column):
    if column == 'target':
        return 'label'
    if column in ['isic_id', 'patient_id']:
        return 'not_feature'
    if column in TRAIN_ONLY_IDDX_FULL_TEXT_COLS:
        return 'oracle'
    if column in REFERENCE_ONLY_COLS:
        return 'reference'
    if column in RELAXED_EXTRA_COLS:
        return 'relaxed'
    if column in SCHEMA_ONLY_CONSTANT_COLS:
        return 'schema_constant'
    if column in STRICT_COLS:
        return 'strict'
    return 'unassigned'

registry_rows = []
for column in df.columns:
    registry_rows.append(
        {
            'column': column,
            'regime': assign_regime(column),
            'regime_display': REGIME_DISPLAY_MAP.get(assign_regime(column), 'Unassigned'),
            'dtype': str(df[column].dtype),
            'null_ratio_pct': round(float(df[column].isna().mean() * 100), 4),
            'n_unique': int(df[column].nunique(dropna=True)),
        }
    )
column_registry_df = pd.DataFrame(registry_rows)
regime_summary_df = column_registry_df['regime_display'].value_counts().rename_axis('regime').to_frame('column_count')
missing_registry_columns = sorted(set(ORDINARY_METADATA_COLS + RELAXED_EXTRA_COLS + TRAIN_ONLY_IDDX_FULL_TEXT_COLS + REFERENCE_ONLY_COLS + ['isic_id', 'patient_id', 'target']) - set(df.columns))

if missing_registry_columns:
    raise KeyError(f'계약 컬럼이 train metadata에 없습니다: {missing_registry_columns}')

display(Markdown('**표 2-a. regime별 컬럼 수**'))
display(regime_summary_df)
display(Markdown('**표 2-b. role registry**'))
display(column_registry_df.sort_values(['regime', 'column']).reset_index(drop=True))

**표 2-a. regime별 컬럼 수**

,column_count
regime,
strict_input,39
Reference only / excluded reference fields,9
Identifier / grouping key,2
Relaxed context optional sensitivity check,2
Label,1
Schema-only constant metadata / excluded from model input,1
Train-only iddx_full text,1


**표 2-b. role registry**

,column,regime,regime_display,dtype,null_ratio_pct,n_unique
0,target,label,Label,int64,0.0000,2
1,isic_id,not_feature,Identifier / grouping key,object,0.0000,401059
2,patient_id,not_feature,Identifier / grouping key,object,0.0000,1042
3,iddx_full,oracle,Train-only iddx_full text,object,0.0000,52
4,iddx_1,reference,Reference only / excluded reference fields,object,0.0000,3
5,iddx_2,reference,Reference only / excluded reference fields,object,99.7337,14
6,iddx_3,reference,Reference only / excluded reference fields,object,99.7345,25
7,iddx_4,reference,Reference only / excluded reference fields,object,99.8626,26
8,iddx_5,reference,Reference only / excluded reference fields,object,99.9998,1
9,lesion_id,reference,Reference only / excluded reference fields,object,94.5001,22058


### 2.1 해석

column registry는 `strict_input` 39개, schema-only constant metadata 1개, reference-only 9개, identifier/grouping key 2개, relaxed context 2개, label 1개, train-only `iddx_full` text 1개로 분리된다.

`Relaxed context`는 active 입력 후보가 아니라 source/context sensitivity check다. `Reference only / excluded reference fields`는 ordinary input 후보가 아니라 accidental leakage를 막기 위한 제외 registry다.

`image_type`은 train 전체에서 상수인 schema-only column이므로 기본 model input에서 제외한다. `iddx_full`만 train-only privileged text candidate로 남긴다. `iddx_1~5`, `mel_*`, `tbp_lv_dnn_lesion_confidence`, `lesion_id`는 분석/감사용 reference로만 둔다.

## 3. `strict_input` 계약

`strict_input`은 `cv_validation`/`test_data`/inference 시점에 요구할 수 있는 ordinary tabular metadata model input이다. 이 장은 컬럼 목록, 상수 컬럼 제외 사유, 최소 결측/자료형 profile만 고정한다.

In [10]:
strict_input_profile_df = column_registry_df.loc[column_registry_df['column'].isin(STRICT_COLS)].copy()
strict_input_profile_df['input_contract'] = 'ordinary inference-time tabular input'
schema_constant_profile_df = column_registry_df.loc[column_registry_df['column'].isin(SCHEMA_ONLY_CONSTANT_COLS)].copy()
schema_constant_profile_df['input_contract'] = 'schema-only constant metadata; excluded from strict_input'
strict_input_numeric_cols = [column for column in STRICT_COLS if pd.api.types.is_numeric_dtype(df[column])]
strict_input_categorical_cols = [column for column in STRICT_COLS if column not in strict_input_numeric_cols]
strict_input_summary_df = pd.DataFrame(
    [
        {'item': 'strict_input_columns', 'value': len(STRICT_COLS)},
        {'item': 'schema_only_constant_columns_excluded', 'value': len(SCHEMA_ONLY_CONSTANT_COLS)},
        {'item': 'numeric_columns', 'value': len(strict_input_numeric_cols)},
        {'item': 'categorical_columns', 'value': len(strict_input_categorical_cols)},
        {'item': 'max_null_ratio_pct', 'value': round(float(strict_input_profile_df['null_ratio_pct'].max()), 4)},
    ]
)

display(Markdown('**표 3-a. strict_input 요약**'))
display(strict_input_summary_df)
display(Markdown('**표 3-b. strict_input 컬럼 계약**'))
display(strict_input_profile_df[['column', 'dtype', 'null_ratio_pct', 'n_unique', 'input_contract']].reset_index(drop=True))
display(Markdown('**표 3-c. schema-only constant metadata 제외 계약**'))
display(schema_constant_profile_df[['column', 'dtype', 'null_ratio_pct', 'n_unique', 'input_contract']].reset_index(drop=True))

**표 3-a. strict_input 요약**

,item,value
0,strict_input_columns,39.0000
1,schema_only_constant_columns_excluded,1.0000
2,numeric_columns,34.0000
3,categorical_columns,5.0000
4,max_null_ratio_pct,2.8716


**표 3-b. strict_input 컬럼 계약**

,column,dtype,null_ratio_pct,n_unique,input_contract
0,age_approx,float64,0.6977,16,ordinary inference-time tabular input
1,sex,object,2.8716,2,ordinary inference-time tabular input
2,anatom_site_general,object,1.4352,5,ordinary inference-time tabular input
3,clin_size_long_diam_mm,float64,0.0000,1758,ordinary inference-time tabular input
4,tbp_tile_type,object,0.0000,2,ordinary inference-time tabular input
5,tbp_lv_A,float64,0.0000,386052,ordinary inference-time tabular input
6,tbp_lv_Aext,float64,0.0000,385304,ordinary inference-time tabular input
7,tbp_lv_B,float64,0.0000,389890,ordinary inference-time tabular input
8,tbp_lv_Bext,float64,0.0000,387763,ordinary inference-time tabular input
9,tbp_lv_C,float64,0.0000,390703,ordinary inference-time tabular input


**표 3-c. schema-only constant metadata 제외 계약**

,column,dtype,null_ratio_pct,n_unique,input_contract
0,image_type,object,0.0,1,schema-only constant metadata; excluded from s...


### 3.1 해석

`strict_input`은 원시 metadata 중심의 일반 model input 계약이며, 총 39개 컬럼으로 구성된다. 이 중 numeric column은 34개, categorical column은 5개이고, 최대 결측률은 `sex` 기준 2.8716%이다.

`image_type`은 train 전체에서 `TBP tile: close-up` 하나의 값만 갖는 상수 컬럼이므로 기본 `strict_input`에서 제외한다. 이 제외는 leakage 때문이 아니라 zero-variance column으로서 discriminative information이 없기 때문이다.

이 노트북에서는 새로운 tabular feature를 만들거나 변수 선택을 하지 않는다.

결측 대체, scaling, categorical encoding, missing indicator 같은 preprocessing state는 실제 실험의 `cv_train` 또는 `final_train`에서만 fit해야 한다.

## 4. `iddx_full` train-only 계약

`iddx_full`은 ordinary tabular feature가 아니다. 같은 `isic_id` sample에 연결된 train-only text field로 추적하며, tokenizer/LM encoder 또는 auxiliary target 생성은 train split 안에서만 허용한다.

In [11]:
required_sample_columns = ['isic_id', 'patient_id', 'lesion_id', 'target', *STRICT_COLS, 'iddx_full']
missing_sample_columns = [column for column in required_sample_columns if column not in df.columns]
if missing_sample_columns:
    raise KeyError(f'sample-level contract 컬럼이 없습니다: {missing_sample_columns}')

sample_alignment_check_df = pd.DataFrame(
    [
        {'check': 'isic_id_unique', 'pass': bool(df['isic_id'].is_unique), 'value': int(df['isic_id'].nunique())},
        {'check': 'patient_id_present', 'pass': bool(df['patient_id'].notna().all()), 'value': int(df['patient_id'].isna().sum())},
        {'check': 'target_binary', 'pass': set(df['target'].dropna().unique()).issubset({0, 1}), 'value': sorted(df['target'].dropna().unique().tolist())},
        {'check': 'iddx_full_present', 'pass': bool(df['iddx_full'].notna().all()), 'value': int(df['iddx_full'].isna().sum())},
    ]
)

phase_input_contract_df = pd.DataFrame(
    [
        {
            'phase': 'cv_train',
            'strict_input': 'required',
            'iddx_full_or_derived': 'optional train-only candidate auxiliary signal',
            'allowed_use': 'fit/use candidate auxiliary signal; no exposure outside cv_train/final_train',
        },
        {
            'phase': 'cv_validation',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not used as model input, scoring feature, or threshold feature',
            'allowed_use': 'leakage audit only',
        },
        {
            'phase': 'test_data',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not allowed',
            'allowed_use': 'none',
        },
        {
            'phase': 'inference',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not required',
            'allowed_use': 'none',
        },
    ]
)

display(Markdown('**표 4-a. sample alignment checks**'))
display(sample_alignment_check_df)
display(Markdown('**표 4-b. phase별 입력 계약**'))
display(phase_input_contract_df)

**표 4-a. sample alignment checks**

,check,pass,value
0,isic_id_unique,True,401059
1,patient_id_present,True,0
2,target_binary,True,"[0, 1]"
3,iddx_full_present,True,0


**표 4-b. phase별 입력 계약**

,phase,strict_input,iddx_full_or_derived,allowed_use
0,cv_train,required,optional train-only candidate auxiliary signal,"tokenizer/LM encoder, label embedding target, ..."
1,cv_validation,required,"not used as model input, scoring feature, or threshold feature",leakage audit only
2,test_data,required,not allowed,none
3,inference,required,not required,none


### 4.1 해석

sample alignment check는 모두 통과했다. `isic_id`는 401,059개 unique sample로 확인되며, `patient_id` 결측은 0개, `iddx_full` 결측도 0개이고, `target`은 binary label `{0, 1}`만 포함한다.

`strict_input`과 `iddx_full`은 반드시 같은 `isic_id` row에서 온 값이어야 한다. 다만 같은 row에 존재한다는 사실과 모델 입력으로 사용한다는 사실은 다르다.

`cv_validation`/`test_data`/inference dataloader는 `iddx_full`, `iddx_label_group`, `iddx_label_metric_group` 없이 동작해야 한다.

## 5. Label Embedding / Graph Metric Learning 후보 전처리 계약

이 후보는 `iddx_full`을 직접 inference feature로 쓰지 않고, train-only auxiliary label 또는 prototype/metric target으로 변환하는 방식이다.

표현 컬럼은 `MEL/BCC/SCC/NV/OTHER` 5개 그룹을 유지한다. metric learning active anchor group은 기본적으로 `MEL/BCC/SCC/NV` 4개로 제한한다.

In [12]:
LABEL_EMBEDDING_GROUPS = ['MEL', 'BCC', 'SCC', 'NV', 'OTHER']
LABEL_METRIC_ACTIVE_GROUPS = ['MEL', 'BCC', 'SCC', 'NV']
OTHER_AUX_WEIGHT_DEFAULT = 0.05


def map_iddx_full_to_label_group(value):
    text = str(value)
    if 'Basal cell carcinoma' in text:
        return 'BCC'
    if 'Squamous cell carcinoma' in text:
        return 'SCC'
    if 'Melanoma' in text:
        return 'MEL'
    if text.startswith('Benign::Benign melanocytic proliferations::Nevus'):
        return 'NV'
    return 'OTHER'

iddx_label_profile_df = df[['isic_id', 'patient_id', 'lesion_id', 'target', 'iddx_full']].copy()
iddx_label_profile_df['iddx_label_group'] = iddx_label_profile_df['iddx_full'].map(map_iddx_full_to_label_group)
iddx_label_profile_df['iddx_label_metric_group'] = iddx_label_profile_df['iddx_label_group'].where(
    iddx_label_profile_df['iddx_label_group'].isin(LABEL_METRIC_ACTIVE_GROUPS),
    pd.NA,
)
iddx_label_profile_df['iddx_label_is_metric_anchor'] = iddx_label_profile_df['iddx_label_group'].isin(LABEL_METRIC_ACTIVE_GROUPS)

label_group_summary_df = (
    iddx_label_profile_df.groupby('iddx_label_group', dropna=False)
    .agg(
        rows=('isic_id', 'size'),
        positives=('target', 'sum'),
        patients=('patient_id', 'nunique'),
        unique_iddx_full=('iddx_full', 'nunique'),
        metric_anchor_rows=('iddx_label_is_metric_anchor', 'sum'),
    )
    .reindex(LABEL_EMBEDDING_GROUPS)
)
label_group_summary_df['row_pct'] = (label_group_summary_df['rows'] / len(iddx_label_profile_df) * 100).round(4)
label_group_summary_df['positive_pct_in_group'] = (
    label_group_summary_df['positives'] / label_group_summary_df['rows'].replace(0, np.nan) * 100
).round(4)

label_embedding_column_contract_df = pd.DataFrame(
    [
        {
            'column': 'iddx_label_group',
            'values': 'MEL/BCC/SCC/NV/OTHER',
            'role': '5-way train-only candidate label management unit',
            'fit_scope': 'fixed rule; created only inside candidate pipeline after split assignment after split assignment',
        },
        {
            'column': 'iddx_label_metric_group',
            'values': 'MEL/BCC/SCC/NV or NA',
            'role': 'active metric/prototype anchor group; OTHER masked',
            'fit_scope': 'cv_train/final_train objectives only',
        },
        {
            'column': 'iddx_label_is_metric_anchor',
            'values': 'True for MEL/BCC/SCC/NV, False for OTHER',
            'role': 'supervised contrastive/triplet/prototype anchor mask',
            'fit_scope': 'fixed mask; batch construction cv_train/final_train only',
        },
        {
            'column': 'iddx_label_aux_weight',
            'values': 'class-balanced for active groups; OTHER default 0.0-0.1',
            'role': 'auxiliary CE/prototype loss weight',
            'fit_scope': 'computed from cv_train/final_train only',
        },
    ]
)

mapping_unit_check_df = pd.DataFrame(
    [
        {'iddx_full_example': 'Malignant::Malignant adnexal epithelial proliferations - Follicular::Basal cell carcinoma', 'expected_group': 'BCC'},
        {'iddx_full_example': 'Malignant::Malignant epidermal proliferations::Squamous cell carcinoma in situ', 'expected_group': 'SCC'},
        {'iddx_full_example': 'Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma in situ::Melanoma in situ, associated with a nevus', 'expected_group': 'MEL'},
        {'iddx_full_example': 'Benign::Benign melanocytic proliferations::Nevus::Nevus, Atypical, Dysplastic, or Clark', 'expected_group': 'NV'},
        {'iddx_full_example': 'Benign', 'expected_group': 'OTHER'},
    ]
)
mapping_unit_check_df['mapped_group'] = mapping_unit_check_df['iddx_full_example'].map(map_iddx_full_to_label_group)
mapping_unit_check_df['pass'] = mapping_unit_check_df['mapped_group'] == mapping_unit_check_df['expected_group']

if not mapping_unit_check_df['pass'].all():
    raise AssertionError('iddx_full manual group mapping check failed')

display(Markdown('**표 5-a. iddx_full label group summary**'))
display(label_group_summary_df)
display(Markdown('**표 5-b. label embedding candidate column contract**'))
display(label_embedding_column_contract_df)
display(Markdown('**표 5-c. manual mapping unit check**'))
display(mapping_unit_check_df)

**표 5-a. iddx_full label group summary**

,rows,positives,patients,unique_iddx_full,metric_anchor_rows,row_pct,positive_pct_in_group
iddx_label_group,,,,,,,
MEL,157,157,133,11,157,0.0391,100.0
BCC,163,163,110,4,163,0.0406,100.0
SCC,73,73,48,5,73,0.0182,100.0
NV,443,0,334,11,443,0.1105,0.0
OTHER,400223,0,1041,21,0,99.7916,0.0


**표 5-b. label embedding candidate column contract**

,column,values,role,fit_scope
0,iddx_label_group,MEL/BCC/SCC/NV/OTHER,5-way train-only candidate label management unit,fixed rule; created only inside candidate pipe...
1,iddx_label_metric_group,MEL/BCC/SCC/NV or NA,active metric/prototype anchor group; OTHER ma...,cv_train/final_train objectives only
2,iddx_label_is_metric_anchor,"True for MEL/BCC/SCC/NV, False for OTHER",supervised contrastive/triplet/prototype ancho...,fixed mask; batch construction fold-local trai...
3,iddx_label_aux_weight,class-balanced for active groups; OTHER defaul...,auxiliary CE/prototype loss weight,computed from cv_train/final_train only


**표 5-c. manual mapping unit check**

,iddx_full_example,expected_group,mapped_group,pass
0,Malignant::Malignant adnexal epithelial prolif...,BCC,BCC,True
1,Malignant::Malignant epidermal proliferations:...,SCC,SCC,True
2,Malignant::Malignant melanocytic proliferation...,MEL,MEL,True
3,Benign::Benign melanocytic proliferations::Nev...,NV,NV,True
4,Benign,OTHER,OTHER,True


### 5.1 해석 및 후속 구현 방향

5개 그룹은 label embedding 후보의 관리 단위이고, metric learning의 active anchor group은 `MEL/BCC/SCC/NV` 4개로 제한한다. 매핑 결과 `MEL` 157 rows, `BCC` 163 rows, `SCC` 73 rows는 모두 malignant positive이며, `NV` 443 rows와 `OTHER` 400,223 rows는 benign이다.

`OTHER`는 전체 row의 99.7916%를 차지하므로 active disease metric anchor로 쓰기에 부적절한 background/null group이다. supervised contrastive anchor, triplet anchor, clustering centroid 학습 대상에서 제외한다. Dataset contract와 auxiliary CE ablation에서는 `OTHER`를 보존하되, 기본 loss weight는 `0.0~0.1`로 둔다.

후속 후보 구현 순서는 다음과 같이 고정한다.

1. manual group auxiliary CE
2. prototype alignment
3. supervised contrastive / metric learning
4. graph prior regularization

이 후보는 strict baseline을 대체하지 않으며, 반드시 같은 patient-level split에서 strict baseline과 비교한다.

## 6. 논문용 split terminology와 Triple Stratified CV 계약

이 장은 split CSV를 저장하지 않는다. 대신 `isic2024_official_train_metadata`를 `train_validation_data`와 `test_data`로 나누고, `train_validation_data` 내부에서 `cv_train` / `cv_validation` 5-fold CV를 구성하는 계약과 audit을 고정한다.

`isic2024_official_test_metadata`는 sample 수가 3개뿐이므로 평가에 사용하지 않고 schema 확인 전용으로만 둔다.

In [13]:
SPLIT_SEED = 42
TEST_DATA_SIZE = 0.20
CV_FOLDS = 5


def make_sample_count_bin(series, q=5):
    ranked = series.rank(method='first')
    return pd.qcut(ranked, q=q, labels=False, duplicates='drop').astype(int).astype(str)


patient_split_profile_df = (
    df.groupby('patient_id', dropna=False)
    .agg(
        patient_rows=('isic_id', 'size'),
        positive_rows=('target', 'sum'),
        lesion_count=('lesion_id', lambda value: value.dropna().nunique()),
    )
    .reset_index()
)
patient_split_profile_df['has_malignant'] = patient_split_profile_df['positive_rows'] > 0
patient_split_profile_df['sample_count_bin'] = make_sample_count_bin(patient_split_profile_df['patient_rows'])
patient_split_profile_df['positive_row_bin'] = np.select(
    [patient_split_profile_df['positive_rows'].eq(0), patient_split_profile_df['positive_rows'].eq(1)],
    ['zero', 'one'],
    default='multiple',
)
patient_split_profile_df['triple_stratum'] = (
    patient_split_profile_df['has_malignant'].astype(int).astype(str)
    + '|pos=' + patient_split_profile_df['positive_row_bin'].astype(str)
    + '|size=' + patient_split_profile_df['sample_count_bin'].astype(str)
)


def initialize_group_state(n_groups, bin_values):
    return [
        {
            'patient_ids': [],
            'patients': 0,
            'rows': 0.0,
            'positive_rows': 0.0,
            'malignant_patients': 0.0,
            'bin_counts': {bin_value: 0.0 for bin_value in bin_values},
        }
        for _ in range(n_groups)
    ]


def add_patient_to_state(state, patient_row):
    state['patient_ids'].append(patient_row['patient_id'])
    state['patients'] += 1
    state['rows'] += float(patient_row['patient_rows'])
    state['positive_rows'] += float(patient_row['positive_rows'])
    state['malignant_patients'] += float(patient_row['has_malignant'])
    state['bin_counts'][patient_row['sample_count_bin']] += 1.0


def remove_patient_from_state(state, patient_row):
    state['patient_ids'].remove(patient_row['patient_id'])
    state['patients'] -= 1
    state['rows'] -= float(patient_row['patient_rows'])
    state['positive_rows'] -= float(patient_row['positive_rows'])
    state['malignant_patients'] -= float(patient_row['has_malignant'])
    state['bin_counts'][patient_row['sample_count_bin']] -= 1.0


def copy_group_state(groups):
    copied = []
    for state in groups:
        copied.append(
            {
                'patient_ids': list(state['patient_ids']),
                'patients': state['patients'],
                'rows': state['rows'],
                'positive_rows': state['positive_rows'],
                'malignant_patients': state['malignant_patients'],
                'bin_counts': dict(state['bin_counts']),
            }
        )
    return copied


def relative_abs_error(value, target):
    return abs(value - target) / max(abs(target), 1.0)


def triple_balance_score(groups, total, target_ratios, bin_values):
    score = 0.0
    for group_index, state in enumerate(groups):
        target_ratio = target_ratios[group_index]
        score += 4.0 * relative_abs_error(state['patients'], total['patients'] * target_ratio)
        score += 6.0 * relative_abs_error(state['rows'], total['rows'] * target_ratio)
        score += 10.0 * relative_abs_error(state['positive_rows'], total['positive_rows'] * target_ratio)
        score += 8.0 * relative_abs_error(state['malignant_patients'], total['malignant_patients'] * target_ratio)
        for bin_value in bin_values:
            score += 3.0 * relative_abs_error(
                state['bin_counts'][bin_value],
                total['bin_counts'][bin_value] * target_ratio,
            )
    return float(score)


def build_initial_triple_quota_assignment(patient_frame, n_groups, target_ratios, seed):
    rng = np.random.default_rng(seed)
    assignment = {}
    for _, stratum_frame in patient_frame.groupby('triple_stratum', sort=True):
        stratum_frame = stratum_frame.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000))).reset_index(drop=True)
        expected = np.array(target_ratios, dtype=float) * len(stratum_frame)
        quotas = np.floor(expected).astype(int)
        remainder = len(stratum_frame) - int(quotas.sum())
        if remainder > 0:
            fractional_order = np.argsort(-(expected - quotas))
            for group_index in fractional_order[:remainder]:
                quotas[group_index] += 1
        cursor = 0
        for group_index, quota in enumerate(quotas):
            for patient_id in stratum_frame.iloc[cursor: cursor + quota]['patient_id']:
                assignment[patient_id] = group_index
            cursor += quota
    return assignment


def build_group_state_from_assignment(patient_frame, assignment, n_groups, bin_values):
    groups = initialize_group_state(n_groups, bin_values)
    for _, patient_row in patient_frame.iterrows():
        add_patient_to_state(groups[int(assignment[patient_row['patient_id']])], patient_row)
    return groups


def improve_assignment_by_local_moves(patient_frame, assignment, groups, total, target_ratios, bin_values, seed, max_passes=10):
    rng = np.random.default_rng(seed)
    patient_lookup = {row['patient_id']: row for _, row in patient_frame.iterrows()}
    ordered_patient_ids = patient_frame.assign(_random_tie_breaker=rng.random(len(patient_frame))).sort_values(
        ['has_malignant', 'positive_rows', 'patient_rows', 'lesion_count', '_random_tie_breaker'],
        ascending=[False, False, False, False, True],
    )['patient_id'].tolist()
    current_score = triple_balance_score(groups, total, target_ratios, bin_values)
    for _ in range(max_passes):
        improved = False
        for patient_id in ordered_patient_ids:
            patient_row = patient_lookup[patient_id]
            current_group = int(assignment[patient_id])
            best_move = None
            for candidate_group in rng.permutation(len(groups)):
                candidate_group = int(candidate_group)
                if candidate_group == current_group:
                    continue
                candidate_groups = copy_group_state(groups)
                remove_patient_from_state(candidate_groups[current_group], patient_row)
                add_patient_to_state(candidate_groups[candidate_group], patient_row)
                candidate_score = triple_balance_score(candidate_groups, total, target_ratios, bin_values)
                if candidate_score + 1e-12 < current_score:
                    move_key = (candidate_score, candidate_group)
                    if best_move is None or move_key < best_move['key']:
                        best_move = {'key': move_key, 'group': candidate_group, 'groups': candidate_groups}
            if best_move is not None:
                assignment[patient_id] = best_move['group']
                groups = best_move['groups']
                current_score = best_move['key'][0]
                improved = True
        if not improved:
            break
    return assignment, groups, current_score


def assign_patient_groups_balanced(patient_frame, n_groups, target_ratios, seed, candidate_seeds=20):
    if len(target_ratios) != n_groups:
        raise ValueError('target_ratios length must match n_groups')
    if not np.isclose(sum(target_ratios), 1.0):
        raise ValueError('target_ratios must sum to 1.0')

    patient_frame = patient_frame.copy()
    patient_frame['sample_count_bin'] = patient_frame['sample_count_bin'].astype(str)
    bin_values = sorted(patient_frame['sample_count_bin'].unique())
    total = {
        'patients': float(len(patient_frame)),
        'rows': float(patient_frame['patient_rows'].sum()),
        'positive_rows': max(float(patient_frame['positive_rows'].sum()), 1.0),
        'malignant_patients': max(float(patient_frame['has_malignant'].sum()), 1.0),
        'bin_counts': patient_frame['sample_count_bin'].value_counts().reindex(bin_values, fill_value=0).astype(float).to_dict(),
    }
    best = None
    for candidate_seed in range(seed, seed + candidate_seeds):
        assignment = build_initial_triple_quota_assignment(patient_frame, n_groups, target_ratios, candidate_seed)
        groups = build_group_state_from_assignment(patient_frame, assignment, n_groups, bin_values)
        assignment, groups, score = improve_assignment_by_local_moves(
            patient_frame,
            assignment,
            groups,
            total,
            target_ratios,
            bin_values,
            seed=candidate_seed,
        )
        if best is None or score < best['score']:
            best = {'assignment': dict(assignment), 'groups': groups, 'score': score, 'seed': candidate_seed}
    return best['assignment'], best['groups'], best['score'], best['seed']


holdout_assignment, holdout_groups, HOLDOUT_BALANCE_SCORE, HOLDOUT_SELECTED_SEED = assign_patient_groups_balanced(
    patient_split_profile_df,
    n_groups=2,
    target_ratios=[1.0 - TEST_DATA_SIZE, TEST_DATA_SIZE],
    seed=SPLIT_SEED,
)
patient_split_profile_df['holdout_group'] = patient_split_profile_df['patient_id'].map(holdout_assignment)
patient_split_profile_df['locked_split'] = np.where(
    patient_split_profile_df['holdout_group'].eq(1),
    'test_data',
    'train_validation_data',
)
train_validation_patient_set = set(
    patient_split_profile_df.loc[patient_split_profile_df['locked_split'].eq('train_validation_data'), 'patient_id']
)
test_patient_set = set(
    patient_split_profile_df.loc[patient_split_profile_df['locked_split'].eq('test_data'), 'patient_id']
)

train_validation_patient_profile_df = patient_split_profile_df.loc[
    patient_split_profile_df['locked_split'].eq('train_validation_data')
].copy()
cv_assignment, cv_groups, CV_BALANCE_SCORE, CV_SELECTED_SEED = assign_patient_groups_balanced(
    train_validation_patient_profile_df,
    n_groups=CV_FOLDS,
    target_ratios=[1.0 / CV_FOLDS] * CV_FOLDS,
    seed=SPLIT_SEED,
)
patient_split_profile_df['cv_validation_fold'] = patient_split_profile_df['patient_id'].map(cv_assignment)
train_validation_patient_profile_df['cv_validation_fold'] = train_validation_patient_profile_df['patient_id'].map(cv_assignment)


def summarize_rows(mask, split_name, cv_fold=pd.NA):
    rows = df.loc[mask]
    patient_summary = rows.groupby('patient_id')['target'].sum() if len(rows) else pd.Series(dtype=float)
    return {
        'split': split_name,
        'cv_fold': cv_fold,
        'rows': int(len(rows)),
        'patients': int(rows['patient_id'].nunique()),
        'positive_rows': int(rows['target'].sum()),
        'positive_rate_pct': round(float(rows['target'].mean() * 100), 5) if len(rows) else 0.0,
        'patients_with_malignant': int((patient_summary > 0).sum()),
    }


def summarize_split_balance(patient_frame, split_column):
    total_patients = max(float(len(patient_frame)), 1.0)
    total_rows = max(float(patient_frame['patient_rows'].sum()), 1.0)
    total_positive = max(float(patient_frame['positive_rows'].sum()), 1.0)
    total_malignant_patients = max(float(patient_frame['has_malignant'].sum()), 1.0)
    rows = []
    for split_name, split_frame in patient_frame.groupby(split_column, dropna=False):
        rows.append(
            {
                split_column: split_name,
                'patients': int(len(split_frame)),
                'patient_ratio_pct': round(float(len(split_frame) / total_patients * 100), 3),
                'rows': int(split_frame['patient_rows'].sum()),
                'row_ratio_pct': round(float(split_frame['patient_rows'].sum() / total_rows * 100), 3),
                'positive_rows': int(split_frame['positive_rows'].sum()),
                'positive_row_ratio_pct': round(float(split_frame['positive_rows'].sum() / total_positive * 100), 3),
                'malignant_patients': int(split_frame['has_malignant'].sum()),
                'malignant_patient_ratio_pct': round(float(split_frame['has_malignant'].sum() / total_malignant_patients * 100), 3),
            }
        )
    return pd.DataFrame(rows)


def summarize_cv_balance(patient_frame):
    return summarize_split_balance(patient_frame, 'cv_validation_fold').rename(columns={'cv_validation_fold': 'cv_fold'})


locked_split_summary_df = pd.DataFrame(
    [
        summarize_rows(df['patient_id'].isin(train_validation_patient_set), 'train_validation_data'),
        summarize_rows(df['patient_id'].isin(test_patient_set), 'test_data'),
    ]
)
locked_triple_axis_balance_df = summarize_split_balance(patient_split_profile_df, 'locked_split')
cv_validation_triple_axis_balance_df = summarize_cv_balance(train_validation_patient_profile_df)

cv_audit_rows = []
cv_overlap_rows = []
for cv_fold in range(CV_FOLDS):
    cv_validation_patient_set = set(
        train_validation_patient_profile_df.loc[
            train_validation_patient_profile_df['cv_validation_fold'].eq(cv_fold),
            'patient_id',
        ]
    )
    cv_train_patient_set = train_validation_patient_set - cv_validation_patient_set
    cv_audit_rows.append(summarize_rows(df['patient_id'].isin(cv_train_patient_set), 'cv_train', cv_fold))
    cv_audit_rows.append(summarize_rows(df['patient_id'].isin(cv_validation_patient_set), 'cv_validation', cv_fold))
    cv_overlap_rows.append(
        {
            'cv_fold': cv_fold,
            'cv_train_cv_validation_patient_overlap': len(cv_train_patient_set & cv_validation_patient_set),
            'cv_train_test_data_patient_overlap': len(cv_train_patient_set & test_patient_set),
            'cv_validation_test_data_patient_overlap': len(cv_validation_patient_set & test_patient_set),
        }
    )
cv_fold_audit_df = pd.DataFrame(cv_audit_rows)
cv_patient_overlap_audit_df = pd.DataFrame(cv_overlap_rows)

sample_count_bin_audit_df = (
    patient_split_profile_df.groupby(['locked_split', 'sample_count_bin'], dropna=False)
    .agg(patients=('patient_id', 'nunique'), rows=('patient_rows', 'sum'), positive_rows=('positive_rows', 'sum'))
    .reset_index()
)
cv_sample_count_bin_audit_df = (
    train_validation_patient_profile_df.groupby(['cv_validation_fold', 'sample_count_bin'], dropna=False)
    .agg(patients=('patient_id', 'nunique'), rows=('patient_rows', 'sum'), positive_rows=('positive_rows', 'sum'))
    .reset_index()
    .rename(columns={'cv_validation_fold': 'cv_fold'})
)

split_artifact_contract_df = pd.DataFrame(
    [
        {'artifact': 'data/processed/isic2024_strict_model_input.csv', 'role': 'feature table only; split role not embedded'},
        {'artifact': 'data/splits/isic2024_train_validation_test_split_seed42.csv', 'role': 'patient-level train_validation_data/test_data assignment'},
        {'artifact': 'data/splits/isic2024_train_validation_5fold_seed42.csv', 'role': 'patient-level cv_train/cv_validation fold assignment'},
    ]
)

split_terminology_df = pd.DataFrame(
    [
        {'term': 'isic2024_official_train_metadata', 'meaning': 'official train-metadata.csv; split source for the paper dataset'},
        {'term': 'isic2024_official_test_metadata', 'meaning': 'official test-metadata.csv; schema check only, not used for evaluation'},
        {'term': 'train_validation_data', 'meaning': 'patient-level data remaining after locked test_data is removed'},
        {'term': 'test_data', 'meaning': 'locked fixed holdout test; used once for final paper-facing internal test score'},
        {'term': 'cv_train', 'meaning': 'training split inside train_validation_data 5-fold CV'},
        {'term': 'cv_validation', 'meaning': 'validation split inside train_validation_data 5-fold CV'},
        {'term': 'final_train', 'meaning': 'final training data before test_data evaluation; default is train_validation_data'},
    ]
)

leakage_contract_df = pd.DataFrame(
    [
        {
            'contract': 'patient_disjoint_split',
            'rule': 'same patient_id must not appear across train_validation_data/test_data or cv_train/cv_validation',
            'applies_to': 'all paper-facing experiments',
        },
        {
            'contract': 'lesion_alignment',
            'rule': 'lesion_id/isic_id remain attached to the same patient_id row',
            'applies_to': 'dataset export and dataloader',
        },
        {
            'contract': 'strict_preprocessing_train_only',
            'rule': 'imputation, scaling, categorical encoding, missing indicators fit on cv_train or final_train only',
            'applies_to': 'strict_input preprocessing',
        },
        {
            'contract': 'iddx_derived_train_only',
            'rule': 'tokenizer vocab, LM fine-tuning, class weights, prototypes, metric batches fit/use cv_train or final_train only',
            'applies_to': 'iddx_full candidate branch',
        },
        {
            'contract': 'validation_threshold_only',
            'rule': 'threshold and calibration selected on cv_validation, never on test_data',
            'applies_to': 'metrics and reporting',
        },
        {
            'contract': 'inference_no_iddx',
            'rule': 'predict/inference API must not require iddx_full or iddx_label_group',
            'applies_to': 'deployment-facing path',
        },
    ]
)

phase_leakage_checklist_df = pd.DataFrame(
    [
        {'check': 'iddx_full-derived value not used in cv_validation or test_data scoring', 'required': True},
        {'check': 'iddx_label_group not required by inference Dataset item', 'required': True},
        {'check': 'cv_train-local class weight/prototype/metric batch only', 'required': True},
        {'check': 'patient_id overlap audited before trusting any score', 'required': True},
        {'check': 'test_data not used for model choice, threshold selection, or early stopping', 'required': True},
    ]
)

display(Markdown('**표 6-a. split terminology**'))
display(split_terminology_df)
display(Markdown(f'**표 6-b. locked split audit: Triple Stratified local-search objective, base_seed={SPLIT_SEED}, selected_seed={HOLDOUT_SELECTED_SEED}, test_size={TEST_DATA_SIZE}, balance_score={HOLDOUT_BALANCE_SCORE:.6f}**'))
display(locked_split_summary_df)
display(Markdown('**표 6-c. locked split triple-axis balance audit**'))
display(locked_triple_axis_balance_df)
display(Markdown(f'**표 6-d. train_validation_data 5-fold CV audit: Triple Stratified local-search objective, folds={CV_FOLDS}, base_seed={SPLIT_SEED}, selected_seed={CV_SELECTED_SEED}, balance_score={CV_BALANCE_SCORE:.6f}**'))
display(cv_fold_audit_df)
display(Markdown('**표 6-e. cv_validation triple-axis balance audit**'))
display(cv_validation_triple_axis_balance_df)
display(Markdown('**표 6-f. patient overlap audit**'))
display(cv_patient_overlap_audit_df)
display(Markdown('**표 6-g. sample-count bin audit for train_validation_data/test_data**'))
display(sample_count_bin_audit_df)
display(Markdown('**표 6-h. sample-count bin audit for cv_validation folds**'))
display(cv_sample_count_bin_audit_df)
display(Markdown('**표 6-i. recommended split / dataset artifact contract**'))
display(split_artifact_contract_df)
display(Markdown('**표 6-j. leakage contract**'))
display(leakage_contract_df)
display(Markdown('**표 6-k. candidate leakage checklist**'))
display(phase_leakage_checklist_df)

**표 6-a. split terminology**

,term,meaning
0,isic2024_official_train_metadata,official train-metadata.csv; split source for ...
1,isic2024_official_test_metadata,"official test-metadata.csv; schema check only,..."
2,train_validation_data,patient-level data remaining after locked test...
3,test_data,locked fixed holdout test; used once for final...
4,cv_train,training split inside train_validation_data 5-...
5,cv_validation,validation split inside train_validation_data ...
6,final_train,final training data before test_data evaluatio...


**표 6-b. locked split audit: Triple Stratified local-search objective, base_seed=42, selected_seed=57, test_size=0.2, balance_score=0.319262**

,split,cv_fold,rows,patients,positive_rows,positive_rate_pct,patients_with_malignant
0,train_validation_data,<NA>,320877,832,315,0.09817,207
1,test_data,<NA>,80182,210,78,0.09728,52


**표 6-c. locked split triple-axis balance audit**

...

,artifact,role
0,data/processed/isic2024_strict_model_input.csv,feature table only; split role not embedded
1,data/splits/isic2024_train_validation_test_spl...,patient-level train_validation_data/test_data ...
2,data/splits/isic2024_train_validation_5fold_se...,patient-level cv_train/cv_validation fold assi...


**표 6-j. leakage contract**

,contract,rule,applies_to
0,patient_disjoint_split,same patient_id must not appear across train_v...,all paper-facing experiments
1,lesion_alignment,lesion_id/isic_id remain attached to the same ...,dataset export and dataloader
2,strict_preprocessing_train_only,"imputation, scaling, categorical encoding, mis...",strict_input preprocessing
3,iddx_derived_train_only,"tokenizer vocab, LM fine-tuning, class weights...",iddx_full candidate branch
4,validation_threshold_only,threshold and calibration selected on cv_valid...,metrics and reporting
5,inference_no_iddx,predict/inference API must not require iddx_fu...,deployment-facing path


**표 6-k. candidate leakage checklist**

,check,required
0,iddx_full-derived value not used in cv_validat...,True
1,iddx_label_group not required by inference Dat...,True
2,cv_train-local class weight/prototype/metric b...,True
3,patient_id overlap audited before trusting any...,True
4,"test_data not used for model choice, threshold...",True


### 6.1 해석

`isic2024_official_train_metadata`는 논문용 split의 source이고, `isic2024_official_test_metadata`는 schema 확인 전용이다. public test metadata는 sample 수가 3개뿐이므로 internal evaluation에 사용하지 않는다.

먼저 patient-level로 `test_data`를 잠그고, 나머지를 `train_validation_data`로 둔다. 이후 `train_validation_data` 안에서만 5-fold CV를 만들며, CV 내부 split 이름은 `cv_train`과 `cv_validation`으로 제한한다. 최종 test라는 용어는 `test_data`에만 사용한다.

Triple Stratified split은 sparse한 combined class label을 직접 stratify하지 않고, patient-level local-search objective로 구현한다. 이 objective는 target axis(`positive_rows`), malignant patient axis(`has_malignant`), burden axis(`sample_count_bin`과 `patient_rows`)를 동시에 균형화한다.

각 audit 표는 `test_data`와 `train_validation_data`, 그리고 모든 `cv_train` / `cv_validation` 조합 사이의 patient overlap이 0인지 확인한다. `test_data`는 모델 선택, threshold 선택, early stopping, feature selection에 사용하지 않는다.

`iddx_full`을 같은 sample에서 추적하는 것 자체는 데이터셋 계약상 필요하다. 그러나 그 값이나 파생값이 `cv_validation` scoring 또는 `test_data` scoring, inference API에 들어가면 strict inference protocol이 아니다.

## 8. handoff checklist

이 노트북은 Dataset class와 training pipeline 구현을 직접 수행하지 않는다. 다음 구현 단계에서 확인해야 할 계약을 넘긴다.

In [14]:
experiment_contract_df = pd.DataFrame(
    [
        {
            'experiment_group': 'strict_tabular_baseline',
            'train_inputs': 'strict_input',
            'cv_validation_test_data_inference_inputs': 'strict_input',
            'uses_iddx_full': False,
            'paper_role': 'ordinary strict baseline',
        },
        {
            'experiment_group': 'image_strict_multimodal_baseline',
            'train_inputs': 'image + strict_input',
            'cv_validation_test_data_inference_inputs': 'image + strict_input',
            'uses_iddx_full': False,
            'paper_role': 'main multimodal baseline',
        },
        {
            'experiment_group': 'strict_input_iddx_full_candidate',
            'train_inputs': 'strict_input + train-only iddx_full-derived auxiliary signal',
            'cv_validation_test_data_inference_inputs': 'strict_input only for scoring/inference',
            'uses_iddx_full': 'train_only',
            'paper_role': 'privileged supervision candidate, compare against strict baseline',
        },
    ]
)

implementation_checklist_df = pd.DataFrame(
    [
        {'item': 'Dataset item can return isic_id/patient_id/lesion_id/target/strict_input', 'required': True},
        {'item': 'train dataset may expose iddx_full_train_text or iddx_label_group for candidate branch', 'required': True},
        {'item': 'cv_validation, test_data, and inference dataloaders work without iddx_full', 'required': True},
        {'item': 'model.forward/predict does not require iddx_full in inference mode', 'required': True},
        {'item': 'results record config path, split version, threshold source, inference_requires_iddx_full=False', 'required': True},
    ]
)

display(Markdown('**표 7-a. baseline / candidate experiment contract**'))
display(experiment_contract_df)
display(Markdown('**표 7-b. implementation handoff checklist**'))
display(implementation_checklist_df)

**표 7-a. baseline / candidate experiment contract**

,experiment_group,train_inputs,cv_validation_test_data_inference_inputs,uses_iddx_full,paper_role
0,strict_tabular_baseline,strict_input,strict_input,False,ordinary strict baseline
1,image_strict_multimodal_baseline,image + strict_input,image + strict_input,False,main multimodal baseline
2,strict_input_iddx_full_candidate,strict_input + train-only iddx_full-derived au...,strict_input only for scoring/inference,train_only,"privileged supervision candidate, compare agai..."


**표 7-b. implementation handoff checklist**

,item,required
0,Dataset item can return isic_id/patient_id/les...,True
1,train dataset may expose iddx_full_train_text ...,True
2,"cv_validation, test_data, and inference datalo...",True
3,model.forward/predict does not require iddx_fu...,True
4,"results record config path, split version, thr...",True


### 8.1 최종 정리

이 노트북의 최종 결론은 다음과 같다.

- 메인 실험 입력은 `image + strict_input`이다.
- `image_type`은 train 전체에서 상수인 schema-only metadata이므로 기본 `strict_input` model feature에서 제외한다.
- `isic2024_official_train_metadata`에서 patient-level Triple Stratified local-search objective로 `train_validation_data`와 `test_data`를 먼저 분리한다.
- `train_validation_data` 내부에서만 같은 Triple Stratified objective로 `cv_train` / `cv_validation` 5-fold CV를 구성한다.
- 최종 test score는 처음부터 잠근 `test_data`에서 한 번만 보고한다.
- `iddx_full`은 ordinary feature가 아니라 train-only candidate signal이다.
- `iddx_label_group`은 `MEL/BCC/SCC/NV/OTHER`로 관리하되, metric anchor는 `MEL/BCC/SCC/NV`만 기본 사용한다.
- `OTHER`는 background/null group으로 둔다.
- patient-disjoint split, `cv_train`/`final_train`-only preprocessing, `cv_validation`-only threshold selection, locked `test_data`-only final reporting을 만족해야 paper-facing claim으로 쓸 수 있다.
- inference path는 `iddx_full` 또는 그 파생값을 요구하지 않는다.